# Modeling Optimization v2 — Maji Safi River Intelligence

## EY AI & Data Challenge 2026 — Water Quality Prediction

**Best score so far:** 0.3469 (Opt 15: 65% ExtraTrees + 35% RandomForest, 21 features)  
**Target:** 0.9+

### What's new in v2
| Optimization | Strategy | Expected gain |
|:---:|---|:---:|
| Opt 26 | Spatial GroupKFold CV + 12 TerraClimate vars (NEW: q, swe, vap, ws) + multi-buffer Landsat | Foundation |
| Opt 27 | Temporal lagging of 7 climate vars (ppt, pet, soil, aet, def, q, ws) | +0.03–0.05 |
| Opt 28 | Bayesian hyperparameter search (Optuna - 40 trials) | +0.02–0.05 |
| Opt 29 | Seed averaging (10 seeds) | +0.01–0.02 |

### Required input files
Place these in the same directory as this notebook:
```
water_quality_training_dataset.csv          <- labels
submission_template.csv                     <- validation set
terraclimate_features_training_EXTENDED.csv <- 12-variable TerraClimate (from Snowflake)
terraclimate_features_validation_EXTENDED.csv
landsat_features_training_50m.csv           <- Multi-buffer Landsat (from Snowflake)
landsat_features_training_150m.csv
landsat_features_training_200m.csv
landsat_features_training.csv               <- 100m baseline (existing)
```


In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
import os
import time

warnings.filterwarnings('ignore')

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

print("✅ Core imports done")
print("   Checking for optional packages...")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
    print("   ✅ optuna available (Opt 28 will run)")
except ImportError:
    HAS_OPTUNA = False
    print("   ⚠️  optuna not found — run: pip install optuna  (Opt 28 will be skipped)")

---
## Step 1: Load & Merge All Data

In [ ]:
BASE = '.'  # Adjust if running from a different directory

# ── Training labels ────────────────────────────────────────────────────────────
wq = pd.read_csv(f'{BASE}/water_quality_training_dataset.csv')
wq['Sample Date'] = pd.to_datetime(wq['Sample Date'], dayfirst=True, errors='coerce')
targets = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

# ── Landsat 100m (existing baseline) ──────────────────────────────────────────
ls_train = pd.read_csv(f'{BASE}/landsat_features_training.csv')
ls_val   = pd.read_csv(f'{BASE}/landsat_features_validation.csv')
ls_train['Sample Date'] = pd.to_datetime(ls_train['Sample Date'], dayfirst=True, errors='coerce')
ls_val['Sample Date']   = pd.to_datetime(ls_val['Sample Date'],   dayfirst=True, errors='coerce')

# ── TerraClimate EXTENDED ──────────────────────────────────────────────────────
tc_ext_path_train = f'{BASE}/terraclimate_features_training_EXTENDED.csv'
tc_ext_path_val   = f'{BASE}/terraclimate_features_validation_EXTENDED.csv'

if os.path.exists(tc_ext_path_train):
    tc_train = pd.read_csv(tc_ext_path_train)
    tc_val   = pd.read_csv(tc_ext_path_val)
    print(f"✅ Extended TerraClimate loaded  — {tc_train.shape}  vars: {[c for c in tc_train.columns if c not in ['Latitude','Longitude','Sample Date']]}")
else:
    tc_train = pd.read_csv(f'{BASE}/terraclimate_features_training_EXTENDED.csv'  # fallback
                           .replace('EXTENDED', 'training') if os.path.exists(
                           f'{BASE}/terraclimate_features_training.csv') else tc_ext_path_train)
    tc_val   = pd.read_csv(f'{BASE}/terraclimate_features_validation_EXTENDED.csv'
                           .replace('EXTENDED', 'validation') if os.path.exists(
                           f'{BASE}/terraclimate_features_validation.csv') else tc_ext_path_val)
    print("⚠️  Extended TerraClimate not found — using existing 5-variable file")

tc_train['Sample Date'] = pd.to_datetime(tc_train['Sample Date'], dayfirst=True, errors='coerce')
tc_val['Sample Date']   = pd.to_datetime(tc_val['Sample Date'],   dayfirst=True, errors='coerce')

print(f"\nTraining rows: {len(wq)}")
print(f"Validation rows: {pd.read_csv(f'{BASE}/submission_template.csv').shape[0]}")

In [ ]:
# ── Merge training data ────────────────────────────────────────────────────────
merge_keys = ['Latitude', 'Longitude', 'Sample Date']

train = wq.merge(ls_train, on=merge_keys, how='left', suffixes=('', '_ls'))
train = train.merge(tc_train, on=merge_keys, how='left', suffixes=('', '_tc'))

# ── Multi-buffer Landsat (if available) ───────────────────────────────────────
for buf in [50, 150, 200]:
    fpath = f'{BASE}/landsat_features_training_{buf}m.csv'
    if os.path.exists(fpath):
        buf_df = pd.read_csv(fpath)
        buf_df['Sample Date'] = pd.to_datetime(buf_df['Sample Date'], dayfirst=True, errors='coerce')
        # Rename band columns to include buffer tag
        rename = {c: f"{c}_{buf}m" for c in ['nir','green','swir16','swir22','NDMI','MNDWI']}
        buf_df = buf_df.rename(columns=rename)
        train = train.merge(buf_df, on=merge_keys, how='left')
        print(f"  ✅ Merged Landsat {buf}m training")
    else:
        print(f"  ℹ️  Landsat {buf}m training not found — skipping")

print(f"\nMerged training shape: {train.shape}")
print(f"Columns: {list(train.columns)}")

In [ ]:
# ── Merge validation data ──────────────────────────────────────────────────────
val_base = pd.read_csv(f'{BASE}/submission_template.csv')
val_base['Sample Date'] = pd.to_datetime(val_base['Sample Date'], dayfirst=True, errors='coerce')

val = val_base.merge(ls_val,  on=merge_keys, how='left')
val = val.merge(tc_val, on=merge_keys, how='left', suffixes=('', '_tc'))

for buf in [50, 150, 200]:
    fpath = f'{BASE}/landsat_features_validation_{buf}m.csv'
    if os.path.exists(fpath):
        buf_df = pd.read_csv(fpath)
        buf_df['Sample Date'] = pd.to_datetime(buf_df['Sample Date'], dayfirst=True, errors='coerce')
        rename = {c: f"{c}_{buf}m" for c in ['nir','green','swir16','swir22','NDMI','MNDWI']}
        buf_df = buf_df.rename(columns=rename)
        val = val.merge(buf_df, on=merge_keys, how='left')
        print(f"  ✅ Merged Landsat {buf}m validation")

print(f"\nMerged validation shape: {val.shape}")

---
## Step 2: Feature Engineering

In [ ]:
def engineer_features(df):
    """Build the full feature matrix from a merged DataFrame."""
    df = df.copy()
    eps = 1e-10

    # ── Temporal features ─────────────────────────────────────────────────────
    if 'Sample Date' in df.columns:
        df['month']      = df['Sample Date'].dt.month
        df['season']     = (df['month'] % 12 + 3) // 3
        df['day_of_year']= df['Sample Date'].dt.dayofyear

    # ── Landsat Indices (100m baseline) ───────────────────────────────────────
    if 'nir' in df.columns and 'swir16' in df.columns:
        df['NDWI']         = (df['green'] - df['nir'])    / (df['green'] + df['nir'] + eps)
        df['swir_ratio']   = df['swir22'] / (df['swir16'] + eps)
        df['green_nir_ratio'] = df['green'] / (df['nir'] + eps)

    # ── Climate-derived (Core) ───────────────────────────────────────────────
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_range'] = df['tmax'] - df['tmin']
    if 'ppt' in df.columns and 'pet' in df.columns:
        df['water_balance']   = df['ppt'] - df['pet']
        df['aridity_index']   = df['pet'] / (df['ppt'] + eps)

    # ── Climate-derived (Extended) ──────────────────────────────────────────
    if 'aet' in df.columns and 'pet' in df.columns:
        df['aet_efficiency'] = df['aet'] / (df['pet'] + eps)  # how much demand is met
    if 'def' in df.columns and 'ppt' in df.columns:
        df['deficit_ratio']  = df['def'] / (df['ppt'] + eps)  # drought severity
    if 'q' in df.columns and 'ppt' in df.columns:
        df['runoff_ratio']   = df['q'] / (df['ppt'] + eps)    # surface vs deep water infiltration
    if 'vap' in df.columns and 'tmax' in df.columns:
        df['humidity_proxy']  = df['vap'] / (df['tmax'] + eps)

    # ── Multi-buffer Index Differences (capture delta from stream center) ──────
    # If we have both 50m and 200m, the diff says something about catchment land cover
    if 'NDMI_50m' in df.columns and 'NDMI_200m' in df.columns:
        df['NDMI_delta_catchment'] = df['NDMI_50m'] - df['NDMI_200m']

    return df

train_fe = engineer_features(train)
val_fe   = engineer_features(val)
print(f"Feature-engineered training shape: {train_fe.shape}")

In [ ]:
# ── Auto-detect usable numeric feature columns ────────────────────────────────
EXCLUDE = set(['Latitude', 'Longitude', 'Sample Date', 'Location_ID',
               'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])

# All numeric columns not in exclude list
FEATURE_COLS = [
    c for c in train_fe.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE
]

print(f"Total features available: {len(FEATURE_COLS)}")
print("\nFeature list:")
for i, f in enumerate(FEATURE_COLS, 1):
    nan_pct = train_fe[f].isna().mean() * 100
    print(f"  {i:2d}. {f:<35s}  NaN: {nan_pct:.1f}%")

# Warn about high-NaN features
high_nan = [c for c in FEATURE_COLS if train_fe[c].isna().mean() > 0.5]
if high_nan:
    print(f"\n⚠️  High-NaN features (>50% missing — consider dropping): {high_nan}")

In [ ]:
# ── Prepare final feature matrices & target arrays ────────────────────────────

# Location groups for spatial CV
train_fe['Location_ID'] = (
    train_fe['Latitude'].round(4).astype(str) + '_' +
    train_fe['Longitude'].round(4).astype(str)
)
groups = train_fe['Location_ID']

X_train = train_fe[FEATURE_COLS].copy()
X_val   = val_fe[FEATURE_COLS].copy()

# Impute remaining NaNs with column median
for col in FEATURE_COLS:
    med = X_train[col].median()
    X_train[col] = X_train[col].fillna(med)
    X_val[col]   = X_val[col].fillna(med)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)

y_train = train_fe[targets].copy()

# Spatial CV splitter (location-aware)
gkf = GroupKFold(n_splits=5)

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")
print(f"y_train: {y_train.shape}")
print(f"Unique locations: {groups.nunique()}")
print("\n✅ Data ready for modeling")

---
## Opt 26 — Spatial CV Baseline: ET+RF Blend + Extended Features

Architecture identical to Opt 15 (65% ET + 35% RF) but now with:
- Extended TerraClimate (adds aet, def, srad)
- Multi-buffer Landsat features (where available)
- Proper spatial GroupKFold cross-validation (no location leakage)

In [ ]:
print("🔧 OPT 26 — SPATIAL CV BASELINE")
print("=" * 60)

t0 = time.time()

# Model params — same as Opt 15
ET_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', random_state=42, n_jobs=-1)
RF_PARAMS = dict(n_estimators=400, max_depth=20, min_samples_leaf=10,
                 max_features='sqrt', random_state=42, n_jobs=-1)
ET_WEIGHT, RF_WEIGHT = 0.65, 0.35

# Spatial GroupKFold CV
cv_r2_per_target = {t: [] for t in targets}

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train_sc, groups=groups), 1):
    Xtr, Xva = X_train_sc[tr_idx], X_train_sc[va_idx]
    fold_preds = np.zeros((len(va_idx), len(targets)))

    for ti, target in enumerate(targets):
        ytr = y_train[target].values[tr_idx]
        yva = y_train[target].values[va_idx]

        et = ExtraTreesRegressor(**ET_PARAMS)
        rf = RandomForestRegressor(**RF_PARAMS)
        et.fit(Xtr, ytr)
        rf.fit(Xtr, ytr)

        pred = ET_WEIGHT * et.predict(Xva) + RF_WEIGHT * rf.predict(Xva)
        pred = np.clip(pred, 0, None)
        fold_preds[:, ti] = pred
        cv_r2_per_target[target].append(r2_score(yva, pred))

    mean_r2 = np.mean([r2_score(y_train[t].values[va_idx], fold_preds[:, ti])
                       for ti, t in enumerate(targets)])
    print(f"  Fold {fold}: R² = {mean_r2:.4f}")

print("\nSpatial CV R² per target:")
for t in targets:
    scores = cv_r2_per_target[t]
    print(f"  {t:<35s}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

overall_cv = np.mean([np.mean(v) for v in cv_r2_per_target.values()])
print(f"\n  Mean spatial CV R²: {overall_cv:.4f}")
print(f"  Elapsed: {(time.time()-t0)/60:.1f} min")

print("\n📝 NOTE: Leaderboard uses a different metric, but spatial CV R² > 0.40 suggests leaderboard improvement.")

In [ ]:
# ── Train on ALL data and generate Opt 26 submission ─────────────────────────
print("Training final Opt 26 models on full training data...")

opt26_preds = np.zeros((len(X_val_sc), len(targets)))

for ti, target in enumerate(targets):
    y = y_train[target].values
    et = ExtraTreesRegressor(**ET_PARAMS)
    rf = RandomForestRegressor(**RF_PARAMS)
    et.fit(X_train_sc, y)
    rf.fit(X_train_sc, y)
    opt26_preds[:, ti] = ET_WEIGHT * et.predict(X_val_sc) + RF_WEIGHT * rf.predict(X_val_sc)
    print(f"  ✅ {target}")

opt26_preds = np.clip(opt26_preds, 0, None)

sub26 = val_base[['Latitude', 'Longitude', 'Sample Date']].copy()
sub26['Sample Date'] = sub26['Sample Date'].astype(str)
for ti, t in enumerate(targets):
    sub26[t] = opt26_preds[:, ti]

sub26.to_csv('submission_opt26.csv', index=False)
print(f"\n✅ submission_opt26.csv saved — {sub26.shape}")
print(sub26.head())

---
## Opt 27 — Temporal Climate Lagging

Water quality responds to **past** weather, not just the current month.  
A rainfall event 1–3 months ago drives today's nutrient concentrations.

Strategy: add 1-month, 2-month, 3-month lags for `ppt`, `pet`, `soil` (and `aet`, `def` if available).

In [ ]:
print("🔧 OPT 27 — TEMPORAL CLIMATE LAGGING")
print("=" * 60)

# Climate variables to lag (High-impact ones first)
LAG_VARS = [c for c in ['ppt', 'pet', 'soil', 'aet', 'def', 'q', 'ws'] if c in train_fe.columns]
LAG_PERIODS = [1, 2, 3]  # months
print(f"Variables to lag: {LAG_VARS}")
print(f"Lag periods: {LAG_PERIODS} months")

def add_temporal_lags(df, lag_vars, lag_periods, location_col='Location_ID'):
    df = df.copy().sort_values([location_col, 'Sample Date'])
    for var in lag_vars:
        for lag in lag_periods:
            df[f'{var}_lag{lag}'] = df.groupby(location_col)[var].shift(lag)
    return df

train_lag = add_temporal_lags(train_fe, LAG_VARS, LAG_PERIODS)
val_lag   = val_fe.copy()
for var in LAG_VARS:
    for lag in LAG_PERIODS:
        col = f'{var}_lag{lag}'
        med = train_lag[col].median()
        val_lag[col] = med

lag_cols = [f'{v}_lag{l}' for v in LAG_VARS for l in LAG_PERIODS if f'{v}_lag{l}' in train_lag.columns]
FEATURE_COLS_27 = FEATURE_COLS + lag_cols
print(f"\nOpt 27: Added {len(lag_cols)} lag features. Total: {len(FEATURE_COLS_27)}")

In [ ]:
# ── Prep + Spatial CV for Opt 27 ─────────────────────────────────────────────
X27_train = train_lag[FEATURE_COLS_27].fillna(train_lag[FEATURE_COLS_27].median())
X27_val   = val_lag[FEATURE_COLS_27].fillna(train_lag[FEATURE_COLS_27].median())

scaler27 = StandardScaler()
X27_train_sc = scaler27.fit_transform(X27_train)
X27_val_sc   = scaler27.transform(X27_val)

groups27 = train_lag['Location_ID']

t0 = time.time()
cv_r2_27 = {t: [] for t in targets}

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X27_train_sc, groups=groups27), 1):
    fold_preds = np.zeros((len(va_idx), len(targets)))
    for ti, target in enumerate(targets):
        ytr = y_train[target].values[tr_idx]
        yva = y_train[target].values[va_idx]
        et = ExtraTreesRegressor(**ET_PARAMS)
        rf = RandomForestRegressor(**RF_PARAMS)
        et.fit(X27_train_sc[tr_idx], ytr)
        rf.fit(X27_train_sc[tr_idx], ytr)
        pred = ET_WEIGHT * et.predict(X27_val_sc if False else X27_train_sc[va_idx]) + RF_WEIGHT * rf.predict(X27_train_sc[va_idx])
        pred = np.clip(pred, 0, None)
        cv_r2_27[target].append(r2_score(yva, pred))

print("Opt 27 Spatial CV R² per target:")
for t in targets:
    scores = cv_r2_27[t]
    print(f"  {t:<35s}: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

overall_27 = np.mean([np.mean(v) for v in cv_r2_27.values()])
print(f"\n  Mean Opt 27 spatial CV R²: {overall_27:.4f}")
print(f"  vs Opt 26 baseline:        {overall_cv:.4f}  (Δ = {overall_27 - overall_cv:+.4f})")
print(f"  Elapsed: {(time.time()-t0)/60:.1f} min")

BEST_FEATURES = FEATURE_COLS_27 if overall_27 > overall_cv else FEATURE_COLS
BEST_SCALER   = scaler27 if overall_27 > overall_cv else scaler
BEST_X_TRAIN  = X27_train_sc if overall_27 > overall_cv else X_train_sc
BEST_X_VAL    = X27_val_sc   if overall_27 > overall_cv else X_val_sc
BEST_CV       = max(overall_27, overall_cv)
BEST_OPT      = 27 if overall_27 > overall_cv else 26
print(f"\n🏆 Best so far: Opt {BEST_OPT}  (CV R² = {BEST_CV:.4f})")

In [ ]:
# ── Generate Opt 27 submission ─────────────────────────────────────────────────
opt27_preds = np.zeros((len(X27_val_sc), len(targets)))
for ti, target in enumerate(targets):
    y = y_train[target].values
    et = ExtraTreesRegressor(**ET_PARAMS)
    rf = RandomForestRegressor(**RF_PARAMS)
    et.fit(X27_train_sc, y)
    rf.fit(X27_train_sc, y)
    opt27_preds[:, ti] = np.clip(
        ET_WEIGHT * et.predict(X27_val_sc) + RF_WEIGHT * rf.predict(X27_val_sc), 0, None
    )

sub27 = val_base[['Latitude', 'Longitude', 'Sample Date']].copy()
sub27['Sample Date'] = sub27['Sample Date'].astype(str)
for ti, t in enumerate(targets):
    sub27[t] = opt27_preds[:, ti]

sub27.to_csv('submission_opt27.csv', index=False)
print(f"✅ submission_opt27.csv saved")

---
## Opt 28 — Bayesian Hyperparameter Search (Optuna)

23 manual experiments explored a tiny fraction of hyperparameter space.  
Optuna uses **Tree-structured Parzen Estimators (TPE)** to efficiently find optimal params.

Search space: `min_samples_leaf` (5–25), `max_depth` (10–30), `n_estimators` (200–800), `ET/RF blend weight` (0.5–0.9).

In [ ]:
if not HAS_OPTUNA:
    print("⚠️  Optuna not installed — skipping Opt 28.")
    print("   To install: pip install optuna")
else:
    print("🔧 OPT 28 — BAYESIAN HYPERPARAMETER SEARCH (Optuna)")
    print("=" * 60)

    # Use the best feature set from Opt 26 or 27
    X_opt = BEST_X_TRAIN
    y_mean = y_train.mean(axis=1).values  # optimize on mean-target R² for speed

    def objective(trial):
        leaf     = trial.suggest_int('min_samples_leaf', 5, 25)
        depth    = trial.suggest_int('max_depth', 10, 30)
        n_est    = trial.suggest_int('n_estimators', 200, 800, step=100)
        et_w     = trial.suggest_float('et_weight', 0.50, 0.90)

        params = dict(n_estimators=n_est, max_depth=depth,
                      min_samples_leaf=leaf, max_features='sqrt',
                      random_state=42, n_jobs=-1)

        fold_scores = []
        for tr_idx, va_idx in gkf.split(X_opt, groups=groups):
            Xtr, Xva = X_opt[tr_idx], X_opt[va_idx]
            ytr, yva = y_mean[tr_idx], y_mean[va_idx]

            et = ExtraTreesRegressor(**params)
            rf = RandomForestRegressor(**params)
            et.fit(Xtr, ytr)
            rf.fit(Xtr, ytr)
            pred = np.clip(et_w * et.predict(Xva) + (1-et_w) * rf.predict(Xva), 0, None)
            fold_scores.append(r2_score(yva, pred))

        return np.mean(fold_scores)

    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=40, show_progress_bar=True)

    best = study.best_params
    print(f"\n✅ Best params found (trial {study.best_trial.number}):")
    for k, v in best.items():
        print(f"   {k}: {v}")
    print(f"   Best mean spatial CV R²: {study.best_value:.4f}")
    print(f"   vs Opt 26/27 baseline:   {BEST_CV:.4f}  (Δ = {study.best_value - BEST_CV:+.4f})")

In [ ]:
if HAS_OPTUNA and 'study' in dir():
    best = study.best_params
    BEST_ET_PARAMS = dict(n_estimators=best['n_estimators'], max_depth=best['max_depth'],
                          min_samples_leaf=best['min_samples_leaf'], max_features='sqrt',
                          random_state=42, n_jobs=-1)
    BEST_RF_PARAMS = BEST_ET_PARAMS.copy()
    BEST_ET_W = best['et_weight']
    BEST_RF_W = 1 - BEST_ET_W

    opt28_preds = np.zeros((BEST_X_VAL.shape[0], len(targets)))
    for ti, target in enumerate(targets):
        y = y_train[target].values
        et = ExtraTreesRegressor(**BEST_ET_PARAMS)
        rf = RandomForestRegressor(**BEST_RF_PARAMS)
        et.fit(BEST_X_TRAIN, y)
        rf.fit(BEST_X_TRAIN, y)
        opt28_preds[:, ti] = np.clip(
            BEST_ET_W * et.predict(BEST_X_VAL) + BEST_RF_W * rf.predict(BEST_X_VAL), 0, None
        )
        print(f"  ✅ {target}")

    sub28 = val_base[['Latitude', 'Longitude', 'Sample Date']].copy()
    sub28['Sample Date'] = sub28['Sample Date'].astype(str)
    for ti, t in enumerate(targets):
        sub28[t] = opt28_preds[:, ti]
    sub28.to_csv('submission_opt28.csv', index=False)
    print("\n✅ submission_opt28.csv saved")
else:
    print("Opt 28 skipped — submission_opt28.csv not generated")

---
## Opt 29 — Seed Averaging (10 Seeds)

Train the same model with 10 random seeds and average predictions.  
This reduces variance at essentially zero added complexity.  
Uses the best hyperparameters found in Opt 28 (falls back to Opt 26 params if Optuna not run).

In [ ]:
print("🔧 OPT 29 — SEED AVERAGING (10 seeds)")
print("=" * 60)

# Use best params (Opt 28 if available, else Opt 26 defaults)
if HAS_OPTUNA and 'best' in dir():
    use_leaf  = best['min_samples_leaf']
    use_depth = best['max_depth']
    use_nest  = best['n_estimators']
    use_etw   = best['et_weight']
    X_use_tr  = BEST_X_TRAIN
    X_use_val = BEST_X_VAL
    print("Using Opt 28 (Optuna) best params")
else:
    use_leaf  = 10
    use_depth = 20
    use_nest  = 400
    use_etw   = 0.65
    X_use_tr  = X_train_sc
    X_use_val = X_val_sc
    print("Using Opt 26 default params (Optuna not available)")

use_rfw = 1 - use_etw
SEEDS = list(range(10))

t0 = time.time()
all_seed_preds = np.zeros((len(X_use_val), len(targets), len(SEEDS)))

for si, seed in enumerate(SEEDS):
    for ti, target in enumerate(targets):
        y = y_train[target].values
        et = ExtraTreesRegressor(n_estimators=use_nest, max_depth=use_depth,
                                  min_samples_leaf=use_leaf, max_features='sqrt',
                                  random_state=seed, n_jobs=-1)
        rf = RandomForestRegressor(n_estimators=use_nest, max_depth=use_depth,
                                   min_samples_leaf=use_leaf, max_features='sqrt',
                                   random_state=seed, n_jobs=-1)
        et.fit(X_use_tr, y)
        rf.fit(X_use_tr, y)
        all_seed_preds[:, ti, si] = use_etw * et.predict(X_use_val) + use_rfw * rf.predict(X_use_val)

    print(f"  Seed {seed} complete ({(time.time()-t0)/60:.1f} min elapsed)")

# Average across seeds
opt29_preds = np.mean(all_seed_preds, axis=2)
opt29_preds = np.clip(opt29_preds, 0, None)

sub29 = val_base[['Latitude', 'Longitude', 'Sample Date']].copy()
sub29['Sample Date'] = sub29['Sample Date'].astype(str)
for ti, t in enumerate(targets):
    sub29[t] = opt29_preds[:, ti]

sub29.to_csv('submission_opt29.csv', index=False)
print(f"\n✅ submission_opt29.csv saved")
print(f"   Total time: {(time.time()-t0)/60:.1f} min")
print(sub29.head())

---
## Summary & Next Steps

In [ ]:
print("=" * 60)
print("OPTIMIZATION SUMMARY")
print("=" * 60)
print(f"  Opt 26 spatial CV R²: {overall_cv:.4f}  (baseline, extended features)")
print(f"  Opt 27 spatial CV R²: {overall_27:.4f}  (+ temporal lagging)")
if HAS_OPTUNA and 'study' in dir():
    print(f"  Opt 28 spatial CV R²: {study.best_value:.4f}  (+ Optuna tuning)")
print()
print("Submission files created:")
for fname in ['submission_opt26.csv', 'submission_opt27.csv',
              'submission_opt28.csv', 'submission_opt29.csv']:
    if os.path.exists(fname):
        df = pd.read_csv(fname)
        print(f"  ✅ {fname}  {df.shape}")
    else:
        print(f"  ⚠️  {fname} — not generated")

print()
print("SUBMISSION PRIORITY ORDER:")
print("  1. submission_opt29.csv  (seed averaging = most stable)")
print("  2. submission_opt28.csv  (Optuna-tuned, if available)")
print("  3. submission_opt27.csv  (temporal lags)")
print("  4. submission_opt26.csv  (extended features baseline)")